# Imports

In [27]:
# --- Imports ---
import os
import sys

import numpy as np
import pandas as pd
from tqdm import tqdm

# --- Add Paths ---
# Go up TWO levels (project root directory)
project_root = os.path.dirname(os.path.dirname(os.getcwd()))

# Append the new path to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
    print("Project root added to sys.path")
else:
    print("Project root already in sys.path")

Project root already in sys.path


# User Selects Atoms

In [28]:
# get dataframe
from utils import gro_processing

# Get data
df, title, num_atoms, box_dimensions = gro_processing.read_gro("../../data/npt-HK4.gro")
df_gro = gro_processing.dataframe_gro(df, box_dimensions[0], oxygen_midpoints=False)

# df.head()
# df_gro.head()
# title
# num_atoms
# box_dimensions

In [29]:
# parameters
res_id = 1
num_atoms_per_mol = df_gro.loc[(res_id,),:].shape[0]
num_res = df_gro.index.levels[0][-1]

print(f'{num_atoms_per_mol=}')
print(f'{num_res=}')

num_atoms_per_mol=84
num_res=np.int64(1501)


In [30]:
# user selects atoms within a single molecule
from itertools import chain

def extend_input(user_input):
    if '-' in user_input:
        start, end = user_input.split('-')
        return range(int(start), int(end) + 1)
    else:
        return range(int(user_input), int(user_input) + 1 )

# user chooses atoms within molecule
print("Insert atom id for one molecule.")
print("Use the following format: '20-30; 30; 20; 40-60' (for range use '-', for multi-input split using ';' ")
# user_input = input("Enter atom id:")
user_input = "15-20; 30; 31-33"  # for testing purposes
print(f"\nYou entered: '{user_input}'")

# extract atom indices that user selected
try: 
    indices = [extend_input(i) for i in user_input.split(";")]
    indices = np.fromiter(set(chain.from_iterable(indices)), dtype=int) # unique values only
except: 
    print("Invalid input format. Please use the specified format.")

indices


Insert atom id for one molecule.
Use the following format: '20-30; 30; 20; 40-60' (for range use '-', for multi-input split using ';' 

You entered: '15-20; 30; 31-33'


array([32, 33, 15, 16, 17, 18, 19, 20, 30, 31])

In [31]:
# filters all molecules based on user selected atoms
# indices = indices + (num_atoms_per_mol * (res_id - 1)) # filter single atom
indices_all = [indices]

for i in range(1, num_res):
    indices_all.append(indices + i * num_atoms_per_mol)
    
# flatten the list 
indices_all = np.concatenate(indices_all)
display(indices_all)

# extract data
mask = df_gro["atom_id"].isin(indices_all)
new_df = df_gro.loc[mask].copy()
display(new_df)

array([    32,     33,     15, ..., 126020, 126030, 126031],
      shape=(15010,))

res_name  atom_id      x      y      z
res_id atom_name                                       
1      H15            HK4       15  1.626  0.619  0.413
       C28            HK4       16  1.626  0.427  0.321
       C27            HK4       17  1.581  0.293  0.332
       H14            HK4       18  1.638  0.211  0.286
       C32            HK4       19  1.478  0.267  0.421
...                   ...      ...    ...    ...    ...
1501   H16            HK4   126020  0.571  5.503  1.628
       C38            HK4   126030  0.875  5.447  1.747
       H20            HK4   126031  0.784  5.456  1.806
       C14            HK4   126032  0.495  5.717  1.234
       C9             HK4   126033  0.360  5.748  1.260

[15010 rows x 5 columns]

# Centroids

In [74]:
# Finding centroid from atoms in new_df
def minimum_image(dx, box_dimensions):
    k = dx * (1 / box_dimensions) # handles 3-D case correctly
    print(k)
    k.astype(int)
    return dx - k * box_dimensions

def minimum_image_vector(p1, p2, box_dimensions):
    dx = p1 - p2
    return minimum_image(dx, box_dimensions)

In [65]:
temp_df = new_df.loc[(res_id,slice(None)),['x','y','z']]
display(temp_df)

x      y      z
res_id atom_name                     
1      H15        1.626  0.619  0.413
       C28        1.626  0.427  0.321
       C27        1.581  0.293  0.332
       H14        1.638  0.211  0.286
       C32        1.478  0.267  0.421
       H16        1.474  0.165  0.461
       C38        1.163  0.324  0.538
       H20        1.152  0.225  0.494
       C14        1.662  0.482  0.190
       C9         1.618  0.428  0.069

In [75]:
# print(temp_df.shape)
# print(temp_df.shape[0])

temp_np = temp_df.to_numpy()
print(temp_np)

print('--------------')
ref_atom = temp_np[0]
vector_arr = np.zeros_like(temp_np)
for i in range(temp_df.shape[0]):
    vector_arr[i] = minimum_image_vector(temp_np[i], ref_atom, box_dimensions)

display(vector_arr)
centroid = ref_atom + np.mean(vector_arr, axis=0)
print(centroid)

[[1.626 0.619 0.413]
 [1.626 0.427 0.321]
 [1.581 0.293 0.332]
 [1.638 0.211 0.286]
 [1.478 0.267 0.421]
 [1.474 0.165 0.461]
 [1.163 0.324 0.538]
 [1.152 0.225 0.494]
 [1.662 0.482 0.19 ]
 [1.618 0.428 0.069]]
--------------
[0. 0. 0.]
[ 0.         -0.01706973 -0.00817925]
[-0.00400072 -0.02898298 -0.00720129]
[ 0.00106686 -0.03627318 -0.01129092]
[-0.01315792 -0.03129451  0.00071124]
[-0.01351354 -0.0403628   0.00426743]
[-0.04116295 -0.02622693  0.01111311]
[-0.0421409  -0.03502851  0.00720129]
[ 0.00320057 -0.01217996 -0.01982578]
[-0.00071124 -0.01698083 -0.03058327]


array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00, -2.77555756e-17,  0.00000000e+00],
       [-6.93889390e-18, -5.55111512e-17,  0.00000000e+00],
       [ 0.00000000e+00, -5.55111512e-17,  0.00000000e+00],
       [-2.77555756e-17,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00, -5.55111512e-17,  0.00000000e+00],
       [-5.55111512e-17,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00, -5.55111512e-17,  1.38777878e-17],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00, -2.77555756e-17,  0.00000000e+00]])

[1.626 0.619 0.413]


In [37]:
ref_atom = temp_df.iloc[0].to_numpy()
display(ref_atom)

array([1.626, 0.619, 0.413])

In [ ]:
# Finding centroid from atoms in new_df, honestly should combine with the above new_df for loop
from utils import oxygen_midpoints

def minimum_image_vector(p1, p2, box_length):
    dx = p1 - p2
    reciprocal_half_box = 2 / box_length
    return oxygen_midpoints.minimum_image_jit(dx, box_length, reciprocal_half_box)


centroid_df = pd.DataFrame()
for res in range(1, num_res + 1):
    select_res = new_df.loc[new_df["res_id"] == res][['x','y','z']]
    
    ref_atom = select_res.iloc[1].values
    vector_arr = np.empty((len(select_res),3))
    for atom in range(select_res.shape[0]):
        select_atom = select_res.iloc[atom].values
        vector = np.empty(3)
        for i in range(3):
            vector[i] = minimum_image_vector(ref_atom[i],select_atom[i], box_dimensions[0])
        vector_arr[atom] = vector
    
    midpoint = ref_atom + np.mean(vector_arr)
    temp_df = pd.DataFrame(columns= ['x','y','z'], data = midpoint[None,:], index=[res])
    centroid_df = pd.concat([centroid_df,temp_df])

centroid_df